In [11]:
import numpy as np
import gymnasium as gym

import warnings ; warnings.filterwarnings('ignore')

from pprint import pprint
from tqdm import tqdm_notebook as tqdm

from itertools import cycle

import random

np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123)

In [14]:
# P = gym.make('FrozenLake-v1').env.unwrapped.P

env = gym.make('FrozenLake-v1')
P = env.env.unwrapped.P
init_state = env.reset()
goal_state = 6

- The outer dictionary keys are the states
- The inner dictionary keys are the actions.
- The value of the inner dictionary is a list with all possible transitions for that state-action pair
- The transition tuples have four values:
  - the probability of that transition,
  - the next state,
  - the reward, and a flag indicating
  - whether the next state is terminal.

In [15]:
P

{0: {0: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 4, 0, False)],
  1: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 4, 0, False),
   (0.33333333333333337, 1, 0, False)],
  2: [(0.33333333333333337, 4, 0, False),
   (0.3333333333333333, 1, 0, False),
   (0.33333333333333337, 0, 0, False)],
  3: [(0.33333333333333337, 1, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 0, 0, False)]},
 1: {0: [(0.33333333333333337, 1, 0, False),
   (0.3333333333333333, 0, 0, False),
   (0.33333333333333337, 5, 0, True)],
  1: [(0.33333333333333337, 0, 0, False),
   (0.3333333333333333, 5, 0, True),
   (0.33333333333333337, 2, 0, False)],
  2: [(0.33333333333333337, 5, 0, True),
   (0.3333333333333333, 2, 0, False),
   (0.33333333333333337, 1, 0, False)],
  3: [(0.33333333333333337, 2, 0, False),
   (0.3333333333333333, 1, 0, False),
   (0.33333333333333337, 0, 0, False)]},
 2: {0: [(0.33333333333333337, 2, 0

In [16]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [34]:
LEFT, RIGHT, DOWN, UP = range(4)
x=range(4)
x

range(0, 4)

In [29]:
init_state = env.reset()
goal_state = 15

LEFT, RIGHT, DOWN, UP = range(4)

pi = lambda s: {
    0:LEFT, 1:LEFT, 2:LEFT, 3:LEFT,
    4:LEFT, 5:LEFT, 6:LEFT, 7:LEFT,
    8:LEFT, 9:LEFT, 10:LEFT, 11:LEFT,
    12:LEFT, 13:LEFT, 14:LEFT, 15:LEFT
}[s]



print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)


Policy:
| 00      < | 01      < | 02      < | 03      < |
| 04      < |           | 06      < |           |
| 08      < | 09      < | 10      < |           |
|           | 13      < | 14      < |           |


In [24]:
n = [1, 2, 3, 4, 5, 6]
even = lambda x: x % 2 == 0, n
print(list(even[1]))

[1, 2, 3, 4, 5, 6]


In [4]:
len(P) ## number of states

16

<>:1: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
/tmp/ipython-input-3663074689.py:1: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  P[5][5[5]]


TypeError: 'int' object is not subscriptable

## policy evaulation

In [6]:
def policy_eval(pi, P, gamma =1.0, theta =1e-10):
  ## value function initially
  prev_V = np.zeros(len(P))
  while True :
    V = np.zeros(len(P))
    for s in range(len(P)):
      for prob, next_state, reward, done in P[s][pi(s)]:
        V[s]+=prob*(reward +gamma* prev_V[next_state] * (not done))
    if np.max(np.abs(prev_V - V))< theta:
      prev_V = V.copy()
  return V
